# Phase 2 · Notebook 5 — ML feature table

Phase 1 built a flood-risk map by *hand-weighting* factors. Phase 2 lets a machine-learning model (**XGBoost**) learn the weights itself, using the July 2023 SAR flood as training labels.

This notebook prepares the ingredients:
1. compute a richer set of **terrain features** (adding HAND, curvature, local relief to Phase 1's layers),
2. stack them into one aligned multi-band raster (for prediction later),
3. draw a **balanced training sample** of pixels with their flood label, saved as a table.

Phase 1's notebooks and outputs are untouched — this builds on top of them.

## Step 1 — Load Phase 1 layers

In [1]:
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt, minimum_filter, laplace
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name in ("phase2", "notebooks") else Path.cwd()
OUT = REPO / "data" / "outputs"
P2 = REPO / "phase2"

def load(name):
    with rasterio.open(OUT / name) as src:
        return src.read(1).astype("float32"), src.profile, src.transform

flood, profile, transform = load("flood_mask.tif")
dem, _, _ = load("dem.tif")
slope, _, _ = load("slope.tif")
dist, _, _ = load("dist_river.tif")
built, _, _ = load("builtup.tif")
valid = (flood != 255) & np.isfinite(dem)
print("Phase 1 layers loaded:", flood.shape)

Phase 1 layers loaded: (3118, 2228)


## Step 2 — Engineer extra terrain features

Three physically-motivated features that help a model spot flood-prone ground:

- **HAND** (Height Above Nearest Drainage): how high a pixel sits *above the nearest river*. Low HAND = water reaches it easily. We find each pixel's nearest river pixel and subtract its elevation.
- **Curvature**: concave hollows (valleys) collect water; convex ridges shed it. The Laplacian of elevation captures this.
- **Local relief**: how much a pixel sticks up above the lowest point nearby — small relief = a flat basin.

In [2]:
dem_filled = np.nan_to_num(dem, nan=float(np.nanmean(dem)))

# HAND — height above nearest river pixel
rivers = gpd.read_file(OUT / "rivers.geojson")
river_raster = rasterize([(g, 1) for g in rivers.geometry], out_shape=flood.shape,
                         transform=transform, fill=0, all_touched=True, dtype="uint8")
nearest = distance_transform_edt(river_raster == 0, return_distances=False, return_indices=True)
hand = (dem - dem[tuple(nearest)]).astype("float32")

# Curvature and local relief
curvature = laplace(dem_filled).astype("float32")
local_relief = (dem - minimum_filter(dem_filled, size=51)).astype("float32")

features = {
    "elevation": dem, "slope": slope, "dist_river": dist, "builtup": built,
    "hand": hand, "curvature": curvature, "local_relief": local_relief,
}
names = list(features)
print("Features:", names)

Features: ['elevation', 'slope', 'dist_river', 'builtup', 'hand', 'curvature', 'local_relief']


## Step 3 — Save the aligned feature stack

We write all seven features into one multi-band GeoTIFF. Notebook 7 will read this back to predict risk for *every* pixel.

In [3]:
stack_prof = profile.copy()
stack_prof.update(count=len(names), dtype="float32", nodata=None, compress="lzw")
with rasterio.open(OUT / "feature_stack.tif", "w", **stack_prof) as dst:
    for i, n in enumerate(names, start=1):
        dst.write(features[n].astype("float32"), i)
        dst.set_band_description(i, n)
print("Saved feature_stack.tif with bands:", names)

Saved feature_stack.tif with bands: ['elevation', 'slope', 'dist_river', 'builtup', 'hand', 'curvature', 'local_relief']


## Step 4 — Draw a balanced training sample

Only ~5% of pixels flooded, so a naive sample would be 95% “dry” and the model would just predict “dry” for everything. We take an **equal number** of flooded and dry pixels.

We also record each pixel's `row, col` and a **spatial block id** (which cell of a 5×5 grid it falls in). Notebook 6 uses those blocks for *spatial* cross-validation — keeping nearby pixels together so the model can't cheat by memorising the neighbourhood.

In [4]:
rng = np.random.default_rng(0)
flood_px = np.argwhere(valid & (flood == 1))
dry_px = np.argwhere(valid & (flood == 0))
n = min(40000, len(flood_px))
flood_px = flood_px[rng.choice(len(flood_px), n, replace=False)]
dry_px = dry_px[rng.choice(len(dry_px), n, replace=False)]
idx = np.vstack([flood_px, dry_px])
label = np.r_[np.ones(len(flood_px), int), np.zeros(len(dry_px), int)]

bh, bw = flood.shape[0] // 5, flood.shape[1] // 5
block = (idx[:, 0] // bh) * 10 + (idx[:, 1] // bw)

table = pd.DataFrame({n_: features[n_][idx[:, 0], idx[:, 1]] for n_ in names})
table["label"] = label
table["row"] = idx[:, 0]; table["col"] = idx[:, 1]; table["block"] = block
table.to_csv(P2 / "training_data.csv", index=False)
print("Training table:", table.shape, "| flood fraction:", round(table.label.mean(), 2))
table.head()

Training table: (80000, 11) | flood fraction: 0.5


,elevation,slope,dist_river,builtup,hand,curvature,local_relief,label,row,col,block
0,201.466492,1.316581,1596.367554,0.0,2.966492,0.000015,1.518326,1,2011,1216,32
1,207.118347,1.817673,1179.566284,0.0,3.118347,0.248810,2.226669,1,478,719,1
2,207.565247,1.466267,1778.182007,0.0,-2.047363,0.000000,1.322433,1,284,315,0
3,205.055649,0.686078,1837.731445,0.0,4.055649,-0.000015,4.055649,1,1075,780,11
4,216.072372,2.085163,10414.144531,0.0,11.572372,0.133682,2.705093,1,8,1757,3


## Recap

We engineered three new terrain features (HAND, curvature, local relief), saved a 7-band `feature_stack.tif`, and wrote a balanced `training_data.csv` with spatial block ids.

**Next — Notebook 6:** train XGBoost, and validate it *honestly* with spatial cross-validation.